In [6]:
import torch
import sys

sys.path.insert(0, "..")
from testbed.models import Idefics
import config

device = torch.device("cuda:1")
lmm = Idefics(
    config.idefics_9b_path,
    torch_dtype=torch.bfloat16,
).to(device)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [7]:
lmm.model

IdeficsForVisionText2Text(
  (model): IdeficsModel(
    (embed_tokens): IdeficsDecoupledEmbedding(
      num_embeddings=32000, num_additional_embeddings=2, embedding_dim=4096, partially_freeze=False
      (additional_embedding): Embedding(2, 4096)
    )
    (vision_model): IdeficsVisionTransformer(
      (embeddings): IdeficsVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(257, 1280)
      )
      (pre_layrnorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (encoder): IdeficsVisionEncoder(
        (layers): ModuleList(
          (0-31): 32 x IdeficsVisionEncoderLayer(
            (self_attn): IdeficsVisionAttention(
              (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
              (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
              (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
              (out_p

In [2]:
import pytorch_lightning as pl
import datasets
from torch.utils.data import (
    DistributedSampler,
    BatchSampler,
    SequentialSampler,
    RandomSampler,
)
import os
import sys
from pathlib import Path

from testbed.data import prepare_caption_input, prepare_dataloader, prepare_vqa_input
import config
import exp_settings as setting


class ICVDataModule(pl.LightningDataModule):

    def __init__(self, lmm) -> None:
        super().__init__()
        self.lmm = lmm
        if setting.task == "vqa":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "vqav2"),
                split="train",
                data_dir=config.vqav2_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        elif setting.task == "caption":
            self.dataset = datasets.load_dataset(
                os.path.join(config.testbed_dir, "data", "coco"),
                data_dir=config.karpathy_coco_caption_dir,
                images_dir=config.coco_dir,
                trust_remote_code=True,
            )
        self.query_set = self.dataset.select(
            range(setting.num_query_samples)
        )

    def setup(self, stage: str) -> None:
        if stage == "fit" or stage is None:
            if setting.task == "vqa":
                self.dataset = datasets.load_dataset(
                    os.path.join(config.testbed_dir, "data", "vqav2"),
                    split="train",
                    data_dir=config.vqav2_dir,
                    images_dir=config.coco_dir,
                    trust_remote_code=True,
                )
            elif setting.task == "caption":
                self.dataset = datasets.load_dataset(
                    os.path.join(config.testbed_dir, "data", "coco"),
                    data_dir=config.karpathy_coco_caption_dir,
                    images_dir=config.coco_dir,
                    trust_remote_code=True,
                )
            self.query_set = self.dataset.select(
                range(setting.num_query_samples)
            )

    def collate_fn(self, batch):
        """
        Split batch into full context, in-context examples, query and answer, and process them into model inputs.
        """
        if setting.task == "vqa":
            context, images = prepare_vqa_input(
                batch, instruction=setting.vqa_instruction
            )
            # we use the first answer as grounding truth
            answers = [item[-1]["answers"][0]["answer"] for item in batch]
        elif setting.task == "caption":
            context, images = prepare_caption_input(
                batch, instruction=setting.caption_instruction
            )
            answers = [item[-1]["sentences_raw"][0] for item in batch]

        # the last two items (take vqa as an example):
        # [
        #   { "role" : "question"
        #     "content" :  ... },
        #   { "role" : "short answer" }
        # ]
        ice_texts = self.lmm.apply_prompt_template([ctx[:-2] for ctx in context])
        query_texts = self.lmm.apply_prompt_template([ctx[-2:] for ctx in context])

        return {
            "ice_texts": ice_texts,
            "query_texts": query_texts,
            "answers": answers,
            "images": images,
        }

    def train_dataloader(self):
        if self.trainer and self.trainer.world_size > 1:
            samplers = [
                BatchSampler(
                    DistributedSampler(self.dataset, shuffle=False),
                    batch_size=setting.num_shot,
                    drop_last=True,
                ),
                DistributedSampler(self.query_set, shuffle=False),
            ]
        else:
            samplers = [
                BatchSampler(
                    SequentialSampler(self.dataset), batch_size=1, drop_last=True
                ),
                SequentialSampler(self.query_set),
            ]

        return prepare_dataloader(
            [self.dataset, self.query_set],
            2,
            num_per_dataset=[1, 1],
            collate_fn=self.collate_fn,
            samplers=samplers,
            # num_workers=1,
            pin_memory=True,
            shuffle=False,
        )

datamodule = ICVDataModule(lmm)
dataloader = datamodule.train_dataloader()

In [138]:
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torch.optim as optim
from deepspeed.ops.adam import DeepSpeedCPUAdam
from transformers import get_cosine_schedule_with_warmup

import exp_settings as setting
from testbed.models.model_base import HookType


class ICVModel(pl.LightningModule):
    def __init__(self, lmm, icv_encoder: torch.nn.Module) -> None:
        super().__init__()
        self.lmm = lmm
        self.lmm.requires_grad_(False)
        self.icv_encoder = icv_encoder

    def generate_label_mask(self, inputs, num_separator, keep_bos=False):
        """
        Generates label mask which masks tokens before num_separator pad_tokens from given inputs.
        """
        input_ids = inputs["input_ids"]
        batch_size, seq_len = input_ids.shape
        pad_mask = input_ids == self.lmm.processor.tokenizer.pad_token_id
        non_pad_mask = ~pad_mask
        label_mask = torch.zeros_like(input_ids, dtype=torch.bool)
        if self.lmm.processor.tokenizer.padding_side == "left":
            eos_position = non_pad_mask.int().argmax(dim=1)

        for i in range(batch_size):
            seq_pad_positions = pad_mask[i].nonzero(as_tuple=False).squeeze(-1)
            num_pads = len(seq_pad_positions)
            if num_pads < num_separator:
                raise ValueError(
                    f"Sequence {i} has fewer pad tokens ({num_pads}) than num_separator ({num_separator})"
                )

            if self.lmm.processor.tokenizer.padding_side == "left":
                seq_pad_positions = seq_pad_positions[
                    seq_pad_positions > eos_position[i]
                ]

            sep_position = seq_pad_positions[num_separator - 1].item()
            label_mask[i, sep_position + 1 :] = True

        return label_mask & non_pad_mask

    def kl_with_last_logits(self, ice_texts, query_texts, answers, images):
        pad_token, pad_token_id, eos_token = (
            self.lmm.processor.tokenizer.pad_token,
            self.lmm.processor.tokenizer.pad_token_id,
            self.lmm.processor.tokenizer.eos_token,
        )

        hooks = self.icv_encoder.register_hook_for(self.lmm)

        # step 1. ICV + query + [PAD] + answer [EOS] forward process
        query_answer = [
            query + pad_token + answer + eos_token
            for query, answer in zip(query_texts, answers)
        ]
        query_images = [img[-setting.num_image_in_query :] for img in images]
        query_inputs = self.lmm.process_input(query_answer, query_images).to(
            self.device
        )
        query_inputs["attention_mask"] = query_inputs["input_ids"] != pad_token_id
        query_outputs = self.lmm.model(
            **query_inputs,
            labels=query_inputs["input_ids"],
        )
        icv_logits = query_outputs["logits"]
        for hook in hooks:
            hook.remove()

        # step 2. ICE + query + [PAD] answer [EOS] forward process
        full_text = [
            ice + query + pad_token + answer + eos_token
            for ice, query, answer in zip(ice_texts, query_texts, answers)
        ]
        inputs = self.lmm.process_input(full_text, images).to(self.device)
        inputs["attention_mask"] = inputs["input_ids"] != pad_token_id
        with torch.no_grad():
            ice_logits = self.lmm.model(**inputs)["logits"]

        # step 3. extract answer logits & calculate kl divergency
        kl_loss = F.kl_div(
            icv_logits[self.generate_label_mask(query_inputs, 1)].log_softmax(dim=-1),
            ice_logits[self.generate_label_mask(inputs, 1)].softmax(dim=-1),
            reduction="batchmean",
            log_target=False,
        )
        ce_loss = query_outputs["loss"]
        total_loss = kl_loss + setting.ce_loss_weight * ce_loss

        return {
            "kl_loss": kl_loss,
            "ce_loss": ce_loss,
            "loss": total_loss,
        }

    def kl_each_layer(self, ice_texts, query_texts, answers, images):
        pad_token, pad_token_id, bos_token_id, eos_token = (
            self.lmm.processor.tokenizer.pad_token,
            self.lmm.processor.tokenizer.pad_token_id,
            self.lmm.processor.tokenizer.bos_token_id,
            self.lmm.processor.tokenizer.eos_token,
        )

        hooks, record_hooks = self.icv_encoder.register_hook_for(self.lmm)

        # step 1. ICV + query + [PAD] + answer [EOS] forward process
        query_answer = [
            query + pad_token + answer + eos_token
            for query, answer in zip(query_texts, answers)
        ]
        query_images = [img[-setting.num_image_in_query :] for img in images]
        query_inputs = self.lmm.process_input(query_answer, query_images).to(
            self.device
        )
        query_inputs["attention_mask"] = query_inputs["input_ids"] != pad_token_id
        self.lmm.model(**query_inputs)
        query_label_mask = query_inputs["attention_mask"] & (
            query_inputs["input_ids"] != bos_token_id
        )
        icv_hidden_states = torch.cat(
            [hs[query_label_mask] for hs in self.icv_encoder.hidden_states]
        )
        for hook in hooks:
            hook.remove()

        # step 2. ICE [PAD] query [PAD] answer [EOS] forward process
        full_text = [
            ice + pad_token + query + pad_token + answer + eos_token
            for ice, query, answer in zip(ice_texts, query_texts, answers)
        ]
        inputs = self.lmm.process_input(full_text, images).to(self.device)
        inputs["attention_mask"] = inputs["input_ids"] != pad_token_id
        self.lmm.model(**inputs)["logits"]
        # extract query [PAD] answer [EOS] from hidden_states
        query_label_mask = self.generate_label_mask(inputs, 1)
        ice_hidden_states = torch.cat(
            [hs[query_label_mask] for hs in self.icv_encoder.hidden_states]
        )
        for hook in record_hooks:
            hook.remove()

        # step 3. extract answer logits & calculate kl divergency
        layer_kl_loss = F.kl_div(
            icv_hidden_states.log_softmax(dim=-1),
            ice_hidden_states.softmax(dim=-1),
            reduction="batchmean",
            log_target=False,
        )

        return {
            "loss": layer_kl_loss,
        }

    def forward(self, ice_texts, query_texts, answers, images):
        if (
            not hasattr(self.icv_encoder, "hidden_states")
            or self.icv_encoder.hidden_states is None
        ):
            self.kl_with_last_logits(ice_texts, query_texts, answers, images)
        return self.kl_each_layer(ice_texts, query_texts, answers, images)

    def training_step(self, batch, batch_idx):
        loss_dict = self(**batch)

        self.log_dict(loss_dict, sync_dist=True, prog_bar=True)

        for i, alpha in enumerate(self.icv_encoder.alpha):
            self.log(f"alpha/alpha-{i}", alpha.item())

        return loss_dict["loss"]

    def configure_optimizers(self):
        param_dict = {
            pn: p for pn, p in self.icv_encoder.named_parameters() if p.requires_grad
        }

        alpha_params = [p for n, p in param_dict.items() if "alpha" in n]
        non_alpha_params = [p for n, p in param_dict.items() if not "alpha" in n]

        optim_groups = [
            {"params": non_alpha_params, "lr": setting.icv_lr},
            {"params": alpha_params, "lr": setting.alpha_lr},
        ]

        if "deepspeed" in setting.strategy:
            optimizer = DeepSpeedCPUAdam(
                optim_groups,
                weight_decay=setting.weight_decay,
            )
        else:
            optimizer = optim.Adam(
                optim_groups,
                weight_decay=setting.weight_decay,
            )

        step_batches = self.trainer.estimated_stepping_batches
        warmup_steps = setting.warmup_step
        if isinstance(warmup_steps, float):
            warm_steps = warmup_steps * step_batches
        elif isinstance(warmup_steps, int):
            warm_steps = warmup_steps
        else:
            raise ValueError(
                f"the warm_steps should be int or float, but got {type(warmup_steps)}"
            )
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warm_steps, num_training_steps=step_batches
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    def on_save_checkpoint(self, checkpoint):
        checkpoint["state_dict"] = {
            k: v
            for k, v in checkpoint["state_dict"].items()
            if not k.startswith("model")
        }
        return checkpoint

In [139]:
from testbed.models.model_base import HookType
import torch.nn as nn
import re

class GlobalICVEncoder(nn.Module):
    def __init__(
        self, lmm_hidden_dim, lmm_layers, alpha_init_value=0.1, record_hidden_states=False, **kwargs
    ) -> None:
        super().__init__()

        self.alpha = torch.nn.Parameter(
            torch.full((lmm_layers,), fill_value=alpha_init_value)
        )
        self.icv = torch.nn.Parameter(
            torch.empty(lmm_layers, lmm_hidden_dim).normal_(mean=0.0, std=0.01)
        )
        self.hidden_states = [[] for _ in range(lmm_layers)] if record_hidden_states else None 
        

    def register_hook_for(self, lmm, **model_inputs):
        hooks = lmm.register_forward_hook(HookType.TEXT_MODEL_LAYER, self.hook)
        if self.hidden_states:
            return hooks, lmm.register_forward_hook(HookType.TEXT_MODEL_LAYER, self.record_hook)
        return hooks

    def record_hook(self, m, inputs, outputs, module_name, **kwargs):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        if not isinstance(outputs, tuple):
            outputs = (outputs,)
        hidden_states, *rest = outputs
        self.hidden_states[layer_idx] = hidden_states

    def hook(self, m, inputs, outputs, module_name, **kwargs):
        layer_idx = int(re.findall(r"\d+", module_name)[0])
        if not isinstance(outputs, tuple):
            outputs = (outputs,)
        hidden_states, *rest = outputs
        shift = self.alpha[layer_idx] * self.icv[layer_idx]
        shifted_states = hidden_states + shift[None, None, :]
        normalized_states = (
            shifted_states
            / shifted_states.norm(dim=-1, keepdim=True)
            * hidden_states.norm(dim=-1, keepdim=True)
        )
        return normalized_states, *rest


torch.manual_seed(426)
icv_encoder = GlobalICVEncoder(4096, 32, record_hidden_states=True).to(device, torch.float16)
model = ICVModel(lmm, icv_encoder).to(device, torch.float16)

In [132]:
inputs = next(iter(dataloader))
# ice_texts, query_texts, answers, images = inputs.values()
# pad_token, pad_token_id, eos_token = (
#             lmm.processor.tokenizer.pad_token,
#             lmm.processor.tokenizer.pad_token_id,
#             lmm.processor.tokenizer.eos_token,
#         )
# full_text = [
#             ice + pad_token + query + pad_token + answer + eos_token
#             for ice, query, answer in zip(ice_texts, query_texts, answers)
#         ]
# model_inputs = lmm.process_input(full_text, images)
# print(model_inputs["input_ids"][0])
# model.generate_label_mask(model_inputs, 1)[0]

[458752000, 458752000, 458752001, 458752001]


In [140]:
output = model(**inputs)


torch.Size([1216, 4096])


In [7]:
import torch.nn.functional as F
import torch

a = torch.randn((1, 32 * 19, 30))
b = torch.randn((1, 32 * 19, 30))

c = torch.randn((1, 32 * 14, 30))
d = torch.randn((1, 32 * 14, 30))

In [37]:
def cal_kl(src, tgt):
    res = []

    src = src.view(32, -1, 30)
    tgt = tgt.view(32, -1, 30)
    for i in range(32):
        res.append(
            F.kl_div(
                src[i].view(-1, 30).log_softmax(dim=-1),
                tgt[i].view(-1, 30).softmax(dim=-1),
                log_target=False,
                reduction="batchmean",
            )
        )

    return sum(res) / 32


(cal_kl(a, b) + cal_kl(c, d)) / 2

tensor(0.9128)

In [17]:
(
    F.kl_div(
        a.view(-1, 30).log_softmax(dim=-1),
        b.view(-1, 30).softmax(dim=-1),
        reduction="batchmean",
        log_target=False,
    )
    + F.kl_div(
        c.view(-1, 30).log_softmax(dim=-1),
        d.view(-1, 30).softmax(dim=-1),
        reduction="batchmean",
        log_target=False,
    )
) / 2

tensor(0.9128)

In [19]:
F.kl_div(
        e.log_softmax(dim=-1),
        f.softmax(dim=-1),
        reduction="batchmean",
        log_target=False,
    )

tensor(963.7701)

In [12]:
F.kl_div(
    a.log_softmax(dim=-1), b.softmax(dim=-1), reduction="none", log_target=False
).flatten().mean()

tensor(0.0304)